In [32]:
import os
import random
from pathlib import Path
from tqdm import tqdm

import sys
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.llm.llm_runtime import (
    jsonl_append, ensure_dir,
    DiskCache, generate_with_cache_and_retries,
    OllamaChatBackend, OpenAIResponsesBackend, MistralChatBackend
)
from src.llm.summary_schema import parse_structured_summary
from src.utils.json_utils import load_jsonl
from src.llm.llm_runtime import now_ts

DATA_DIR = Path("../data")
PROC_DIR = DATA_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = DATA_DIR / "predictions"
CACHE_DIR = DATA_DIR / ".cache_llm"

DATASET_PATH = PROC_DIR / "work_chats_dataset.jsonl"
GOLD_PATH = PROC_DIR / "gold_summaries.jsonl"

SEED = 42
random.seed(SEED)

ensure_dir(str(OUT_DIR))
ensure_dir(str(CACHE_DIR))

In [19]:
SYSTEM_PROMPT = (
    "Ты помощник для суммаризации рабочих переписок.\n"
    "Пиши строго на русском языке.\n"
    "Выведи ТОЛЬКО валидный JSON без комментариев и без markdown.\n"
    "Строго соблюдай схему:\n"
    "{\n"
    '  "Context": "1–3 предложения о сути обсуждения.",\n'
    '  "Decisions": ["Краткие формулировки явных решений."],\n'
    '  "Actions": [\n'
    '    { "assignee": "Кто или null", "description": "Что сделать", "deadline": "Когда или null" }\n'
    "  ],\n"
    '  "Questions": ["Открытые вопросы без ответа в диалоге."]\n'
    "}\n"
    "Если данных для решений/экшенов/вопросов нет — верни пустые списки.\n"
    "Не выдумывай факты."
)

def make_user_prompt(dialogue: str) -> str:
    return (
        "Суммаризируй диалог в JSON по указанной схеме.\n\n"
        "ДИАЛОГ:\n"
        f"{dialogue}\n"
    )

In [30]:
# =========
# LLM backends + paths registry
# =========
MODEL_SPECS = {
    # ---- Local (Ollama) ----
    "qwen2.5_3b_local": {
        "backend": OllamaChatBackend(model="qwen2.5:3b-instruct"),
        "tag": "ollama_qwen2.5_3b",
    },
    "llama3.1_8b_local": {
        "backend": OllamaChatBackend(model="llama3.1:8b"),
        "tag": "ollama_llama3.1_8b",
    },
    "mistral_7b_local": {
        "backend": OllamaChatBackend(model="mistral:7b-instruct"),
        "tag": "ollama_mistral_7b",
    },

    # ---- API ----
    "gpt4o": {
        "backend": OpenAIResponsesBackend(model="gpt-4o"),
        "tag": "openai_gpt4o",
    },
    "mistral_large": {
        "backend": MistralChatBackend(model="mistral-large-latest"),
        "tag": "mistral_large",
    },
}

BACKENDS = {k: v["backend"] for k, v in MODEL_SPECS.items()}

PATHS = {
    k: {
        "tag": v["tag"],
        "cache_path": CACHE_DIR / f'{v["tag"]}.json',
        "out_path": OUT_DIR / f'{v["tag"]}.jsonl',
    }
    for k, v in MODEL_SPECS.items()
}

RUNTIMES = {
    k: {
        "backend": BACKENDS[k],
        "cache": DiskCache(str(PATHS[k]["cache_path"])),
        "out_path": PATHS[k]["out_path"],
        "tag": PATHS[k]["tag"],
    }
    for k in MODEL_SPECS.keys()
}

In [21]:
# =========
# Choose model
# =========
MODEL_KEY = "gpt4o"  # "mistral_large" | "qwen2.5_3b_local" | "llama3.1_8b_local"

backend = RUNTIMES[MODEL_KEY]["backend"]
cache = RUNTIMES[MODEL_KEY]["cache"]
out_path = RUNTIMES[MODEL_KEY]["out_path"]
MODEL_TAG = RUNTIMES[MODEL_KEY]["tag"]

In [22]:
# =========
# Resume logic
# =========
done_ids = set()
if out_path.exists():
    for row in load_jsonl(out_path):
        did = row.get("dialogue_id")
        if did is not None:
            done_ids.add(did)

dataset = load_jsonl(DATASET_PATH)
todo = [x for x in dataset if x.get("dialogue_id") not in done_ids]

print(f"Loaded dataset: {len(dataset)} | already done: {len(done_ids)} | to do: {len(todo)}")

Loaded dataset: 177 | already done: 0 | to do: 177


In [23]:
done_ids = set()
if out_path.exists():
    for row in load_jsonl(out_path):
        did = row.get("dialogue_id")
        if did is not None:
            done_ids.add(did)

dataset = load_jsonl(GOLD_PATH)
todo = [x for x in dataset if x.get("dialogue_id") not in done_ids]

print(f"Loaded gold samples: {len(dataset)} | already done: {len(done_ids)} | to do: {len(todo)}")

Loaded gold samples: 30 | already done: 0 | to do: 30


In [31]:
# =========
# Smoke test: one dialogue for each backend (correct signature)
# =========

assert len(dataset) > 0, "Dataset is empty, nothing to test"

test_sample = todo[0] if todo else dataset[0]
test_dialogue_id = test_sample.get("dialogue_id")
test_dialogue = test_sample.get("dialogue") or ""

test_user_prompt = make_user_prompt(test_dialogue)

print(f"\n🔍 Smoke test on dialogue_id={test_dialogue_id}\n")

for model_key, rt in RUNTIMES.items():
    print(f"--- Testing backend: {model_key} ({rt['tag']}) ---")

    try:
        # отдельный временный кеш, чтобы не трогать основной
        tmp_cache = DiskCache(str(CACHE_DIR / f"_smoke_test_{rt['tag']}.json"))

        gen = generate_with_cache_and_retries(
            backend=rt["backend"],
            cache=tmp_cache,
            system_prompt=SYSTEM_PROMPT,
            user_prompt=test_user_prompt,
            max_retries=2,
            retry_base_s=1.0,
            retry_cap_s=6.0,
            temperature=0.0,
            max_output_tokens=300,  # быстрее для smoke-test
        )

        summary, parse_error = parse_structured_summary(gen.raw_text)

        if gen.error:
            print("❌ CALL ERROR:", gen.error)
        elif parse_error:
            print("⚠️ PARSE ERROR:", parse_error)
            print("raw head:", (gen.raw_text or "")[:200])
        else:
            print("✅ OK")
            print("latency_s:", gen.latency_s, "| cache_hit:", gen.cache_hit)
            print("Context head:", (summary.get("Context") or "")[:120])
            print(
                "Counts:",
                {
                    "Decisions": len(summary.get("Decisions", [])),
                    "Actions": len(summary.get("Actions", [])),
                    "Questions": len(summary.get("Questions", [])),
                }
            )

        # на всякий случай сбросим временный кеш
        tmp_cache.flush()

    except Exception as e:
        print("❌ FAILED (exception)")
        print(type(e).__name__, str(e))

    print()


🔍 Smoke test on dialogue_id=Chat_001__batch_00001

--- Testing backend: qwen2.5_3b_local (ollama_qwen2.5_3b) ---
✅ OK
latency_s: 0.0 | cache_hit: True
Context head: Диалог касается создания столбцов для сбора информации о болевых точках и формирования планов на работу.
Counts: {'Decisions': 2, 'Actions': 1, 'Questions': 1}

--- Testing backend: llama3.1_8b_local (ollama_llama3.1_8b) ---
✅ OK
latency_s: 0.0 | cache_hit: True
Context head: Дискуссия об оценке болей на.scale с 1 по 5.
Counts: {'Decisions': 2, 'Actions': 1, 'Questions': 2}

--- Testing backend: mistral_7b_local (ollama_mistral_7b) ---
✅ OK
latency_s: 15.472429037094116 | cache_hit: False
Context head: Дискуссия о поиске наиболее частых болей и создании столбцов для этого
Counts: {'Decisions': 2, 'Actions': 2, 'Questions': 2}

--- Testing backend: gpt4o (openai_gpt4o) ---
✅ OK
latency_s: 0.0 | cache_hit: True
Context head: Обсуждение процесса оценки проблем и подготовки к встрече с ментором. Участники обсуждают, как фиксир

In [33]:
# =========
# Run ALL models
# =========

for model_key, rt in RUNTIMES.items():
    backend = rt["backend"]
    cache = rt["cache"]
    out_path = rt["out_path"]
    MODEL_TAG = rt["tag"]

    ensure_dir(out_path.parent)
    ensure_dir(CACHE_DIR)

    # ---------
    # Per-model resume: compute done_ids FROM THIS model's out_path
    # ---------
    done_ids = set()
    if out_path.exists():
        for row in load_jsonl(out_path):
            did = row.get("dialogue_id")
            if did is not None:
                done_ids.add(did)

    todo_model = [x for x in todo if x.get("dialogue_id") not in done_ids]

    print(
        f"\n=== MODEL: {model_key} | tag={MODEL_TAG} | backend={backend.name} ===\n"
        f"already done: {len(done_ids)} | to do now: {len(todo_model)} | out: {out_path}\n"
    )

    # ---------
    # Generation loop (per model)
    # ---------
    for item in tqdm(todo_model, desc=f"Generating {MODEL_TAG}"):
        dialogue_id = item.get("dialogue_id")
        dialogue = item.get("dialogue") or ""

        user_prompt = make_user_prompt(dialogue)

        # Params split: Ollama vs API
        is_ollama = backend.name.lower().startswith("ollama")

        try:
            gen = generate_with_cache_and_retries(
                backend=backend,
                cache=cache,
                system_prompt=SYSTEM_PROMPT,
                user_prompt=user_prompt,
                max_retries=4,
                retry_base_s=1.0,
                retry_cap_s=20.0,
                temperature=0.0,
                max_output_tokens=None if is_ollama else 900,
                options={"temperature": 0.0, "num_predict": 900} if is_ollama else None,
            )

            raw_text = gen.raw_text
            latency_s = gen.latency_s
            cache_hit = gen.cache_hit
            call_error = gen.error

        except Exception as e:
            # если случился неожиданный exception (не внутри gen.error)
            raw_text = ""
            latency_s = None
            cache_hit = False
            call_error = f"{type(e).__name__}: {e}"

        summary, parse_error = parse_structured_summary(raw_text)

        row = {
            "dialogue_id": dialogue_id,
            "model": backend.name,
            "model_key": model_key,
            "model_tag": MODEL_TAG,
            "created_at": now_ts(),
            "latency_s": latency_s,
            "cache_hit": cache_hit,
            "call_error": call_error,
            "parse_error": parse_error,
            "dialogue": dialogue,
            "raw_summary": raw_text,
            "summary": summary,
        }

        jsonl_append(str(out_path), row)

        # flush cache periodically
        if random.random() < 0.05:
            cache.flush()

    cache.flush()
    print(f"✅ Saved predictions to: {out_path}")


=== MODEL: qwen2.5_3b_local | tag=ollama_qwen2.5_3b | backend=ollama:qwen2.5:3b-instruct ===
already done: 0 | to do now: 30 | out: ../data/predictions/ollama_qwen2.5_3b.jsonl



Generating ollama_qwen2.5_3b: 100%|██████████| 30/30 [03:32<00:00,  7.09s/it]


✅ Saved predictions to: ../data/predictions/ollama_qwen2.5_3b.jsonl

=== MODEL: llama3.1_8b_local | tag=ollama_llama3.1_8b | backend=ollama:llama3.1:8b ===
already done: 0 | to do now: 30 | out: ../data/predictions/ollama_llama3.1_8b.jsonl



Generating ollama_llama3.1_8b: 100%|██████████| 30/30 [05:46<00:00, 11.54s/it]


✅ Saved predictions to: ../data/predictions/ollama_llama3.1_8b.jsonl

=== MODEL: mistral_7b_local | tag=ollama_mistral_7b | backend=ollama:mistral:7b-instruct ===
already done: 0 | to do now: 30 | out: ../data/predictions/ollama_mistral_7b.jsonl



...

✅ Saved predictions to: ../data/predictions/ollama_mistral_7b.jsonl

=== MODEL: gpt4o | tag=openai_gpt4o | backend=openai:gpt-4o ===
already done: 0 | to do now: 30 | out: ../data/predictions/openai_gpt4o.jsonl



Generating openai_gpt4o: 100%|██████████| 30/30 [02:02<00:00,  4.07s/it]


✅ Saved predictions to: ../data/predictions/openai_gpt4o.jsonl

=== MODEL: mistral_large | tag=mistral_large | backend=mistral:mistral-large-latest ===
already done: 0 | to do now: 30 | out: ../data/predictions/mistral_large.jsonl



Generating mistral_large: 100%|██████████| 30/30 [03:42<00:00,  7.42s/it]

✅ Saved predictions to: ../data/predictions/mistral_large.jsonl
